# 01 Dataset Preparation

This notebook prepares the common dataset inputs used by the later RADAR–LiDAR velocity analysis.

It includes:

- filtering the TruckScenes metadata according to locally available sensor files;
- building key-frame sensor observation data;
- separating RADAR and LiDAR observations;
- building the annotation-derived reference velocity.

The original analysis logic is retained from `condition_velocity.ipynb`.

Reusable outputs are saved to `df/condition/` so that later notebooks can run independently without repeating this preparation stage.

In [1]:
import sys
sys.path.append("../..")

from pathlib import Path

from truckscenes import TruckScenes

from src.config import DATA_ROOT, METADATA_ROOT
from src.sensor_observation import build_sensor_data_df
from src.annotation_velocity import build_vehicle_reference_df
from src.dataset_filtering import prepare_available_sample_data

## 1. Dataset Preparation

### Build Sensor Observation Data

In [2]:
from src.config import DATA_ROOT, VERSION

trucksc = TruckScenes(version=VERSION, dataroot=str(DATA_ROOT))

Loading truckscenes tables for version v1.2-trainval...
11 attribute,
18 calibrated_sensor,
27 category,
1200243 ego_motion_cabin,
1200183 ego_motion_chassis,
1201993 ego_pose,
43769 instance,
23902 sample,
819536 sample_annotation,
2586264 sample_data,
598 scene,
18 sensor,
4 visibility,
598 weather_annotation,
Done loading in 38.882 seconds.
Reverse indexing ...
Done reverse indexing in 21.9 seconds.
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


### Filter Locally Available Sensor Data

Filter the metadata according to the sensor files that are actually available in the local dataset before building the sensor observation tables.

In [3]:
available_sample_data, summary, extra_files = prepare_available_sample_data()
display(summary)

sensor_data_df = build_sensor_data_df(
    metadata_root=METADATA_ROOT,
    sample_data_df=available_sample_data,
)

keyframe_df = sensor_data_df[sensor_data_df["Is Key Frame"].eq(True)].copy()
radar_observation_df = keyframe_df[keyframe_df["Modality"].eq("RADAR")].reset_index(drop=True)
lidar_observation_df = keyframe_df[keyframe_df["Modality"].eq("LiDAR")].reset_index(drop=True)

Metadata records                    2586264
Available local files                820213
Unavailable locally                 1766051
Extra / unreferenced local files          2
Name: Dataset Filtering Summary, dtype: int64

In [4]:
if extra_files:
    print("First extra files:")
    print(*list(extra_files)[:20], sep="\n")

First extra files:
sweeps/CAMERA_LEFT_FRONT/CAMERA_LEFT_FRONT_1707237950165000.jpg
sweeps/CAMERA_LEFT_FRONT/CAMERA_LEFT_FRONT_1707237950665534.jpg


In [5]:
vehicle_reference_df = build_vehicle_reference_df(
    trucksc.sample,
    trucksc.sample_annotation,
    trucksc.instance,
    trucksc.category,
)

print("Reference intervals:", len(vehicle_reference_df))

Reference intervals: 534039


In [ ]:
RESULT_DIR = Path("../../df/condition")

available_sample_data.to_csv(RESULT_DIR / "available_sample_data.csv", index=False)
keyframe_df.to_csv(RESULT_DIR / "keyframe_df.csv", index=False)
radar_observation_df.to_csv(RESULT_DIR / "radar_observation_df.csv", index=False)
lidar_observation_df.to_csv(RESULT_DIR / "lidar_observation_df.csv", index=False)

In [7]:
vehicle_reference_df.to_csv(RESULT_DIR / "vehicle_reference_df.csv", index=False)